In [ ]:
# ============================================================
# Module 0. Imports, random seed and publication-style settings
# ============================================================
import os
import re
import math
import random
import warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.stats import shapiro, jarque_bera, skew, kurtosis
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx

try:
    import lingam
    LINGAM_AVAILABLE = True
except Exception as e:
    LINGAM_AVAILABLE = False
    print("[Warning] lingam is not available. Install it with: pip install lingam")
    print(e)

try:
    from econml.dml import CausalForestDML
    ECONML_AVAILABLE = True
except Exception as e:
    ECONML_AVAILABLE = False
    print("[Warning] econml is not available. DML-CF will be skipped unless installed with: pip install econml")
    print(e)

warnings.filterwarnings("ignore")

SEED = 66
random.seed(SEED)
np.random.seed(SEED)

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "savefig.dpi": 600,
    "figure.dpi": 150,
})

PALETTE = {
    "blue": "#2A6FBB",
    "cyan": "#45A6B8",
    "teal": "#2F8F83",
    "orange": "#E5863D",
    "red": "#B94E48",
    "purple": "#7A5BA6",
    "grey": "#6F7C85",
    "light_grey": "#EAECEF",
    "dark": "#243447",
    "green": "#3B8C6E"
}

OUT_DIR = Path("reviewer3_DAG_adjustment_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("Environment ready.")


In [ ]:
# ============================================================
# Module 1. User-defined data path, variables and column mapping
# ============================================================

DATA_PATH = ""

TARGET = "


LINGAM_EDGE_THRESHOLD = 0.05
RUN_DML_CF = True
DML_MAIN_N_ESTIMATORS = 400
DML_MAIN_MIN_SAMPLES_LEAF = 12
BLOCK_BOOTSTRAP_N_REPS = 80
BLOCK_BOOTSTRAP_N_ESTIMATORS = 200
BLOCK_BOOTSTRAP_BLOCK_SIZE = 14

print("Configuration loaded.")



In [ ]:
# ============================================================
# Module 2. Data loading, column mapping and cleaning
# ============================================================

def normalise_colname(x):
    """Normalise column names for safer matching."""
    s = str(x).strip()
    s = s.replace("\ufeff", "")
    s = re.sub(r"\s+", " ", s)
    return s

def canonical_key(x):
    """Create a loose-matching key for column names."""
    s = normalise_colname(x).lower()
    s = s.replace("\uff08", "(").replace("\uff09", ")")
    s = re.sub(r"[\s_\-\/]+", "", s)
    return s

def read_data_file(data_path):
    """Read csv/xlsx/xls with common encodings."""
    path = Path(data_path)
    if not path.exists():
        raise FileNotFoundError(f"Cannot find data file: {data_path}")

    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".csv":
        encodings = ["utf-8-sig", "utf-8", "gbk", "gb18030", "latin1"]
        last_error = None
        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc)
            except Exception as e:
                last_error = e
        raise last_error

    raise ValueError(f"Unsupported file type: {suffix}. Please use csv, xlsx or xls.")

def build_safe_rename_map(raw_columns, rename_map):
    """Build rename map using exact match first and loose canonical match second."""
    raw_columns = [normalise_colname(c) for c in raw_columns]
    mapping = {}

    for c in raw_columns:
        if c in rename_map:
            mapping[c] = rename_map[c]

    alias_by_key = {canonical_key(k): v for k, v in rename_map.items()}
    for c in raw_columns:
        if c in mapping:
            continue
        key = canonical_key(c)
        if key in alias_by_key:
            mapping[c] = alias_by_key[key]

    return mapping

def make_column_diagnostics(raw_columns, mapped_columns, selected_features):
    rows = []
    for target_col in selected_features:
        status = "matched" if target_col in set(mapped_columns) else "missing"
        matched_raw = []
        for raw, mapped in zip(raw_columns, mapped_columns):
            if mapped == target_col:
                matched_raw.append(raw)
        rows.append({
            "Required variable": target_col,
            "Status": status,
            "Matched raw column(s)": "; ".join(matched_raw) if matched_raw else ""
        })
    diag = pd.DataFrame(rows)
    diag.to_csv(OUT_DIR / "Column_name_matching_diagnostics.csv", index=False, encoding="utf-8-sig")
    return diag

def load_and_prepare_data(data_path=DATA_PATH,
                          rename_map=RENAME_MAP,
                          selected_features=SELECTED_FEATURES):
    """
    Load data, rename columns to manuscript variable names, keep selected variables,
    coerce numeric values, and impute missing values by median.
    """
    df_raw = read_data_file(data_path)
    df_raw.columns = [normalise_colname(c) for c in df_raw.columns]

    safe_map = build_safe_rename_map(df_raw.columns, rename_map)
    df = df_raw.rename(columns=safe_map).copy()
    df.columns = [normalise_colname(c) for c in df.columns]

    diag = make_column_diagnostics(df_raw.columns, df.columns, selected_features)
    missing_cols = [c for c in selected_features if c not in df.columns]

    if missing_cols:
        pd.DataFrame({
            "Missing required variable": missing_cols,
            "Action needed": [
                "Add the corresponding original column name to RENAME_MAP, or check whether this variable was generated in preprocessing."
                for _ in missing_cols
            ]
        }).to_csv(OUT_DIR / "Missing_column_suggestions.csv", index=False, encoding="utf-8-sig")

        print("Column matching diagnostics:")
        display(diag)
        print("Available columns after mapping:")
        display(pd.DataFrame({"Available columns": df.columns.tolist()}))
        raise KeyError(
            "Some required variables are missing after column mapping. "
            "Please check reviewer3_DAG_adjustment_outputs/Missing_column_suggestions.csv "
            "and add the corresponding raw names into RENAME_MAP. Missing variables: "
            + str(missing_cols)
        )

    df = df[selected_features].copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(axis=0, how="all")
    imputer = SimpleImputer(strategy="median")
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns, index=df.index)

    nunique = df_imputed.nunique(dropna=True)
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        print("Dropped constant columns:", constant_cols)
        df_imputed = df_imputed.drop(columns=constant_cols)

    if TARGET not in df_imputed.columns:
        raise KeyError(f"Outcome variable {TARGET} is missing after cleaning.")

    print(f"Data shape after preparation: {df_imputed.shape}")
    display(df_imputed.head())
    return df_imputed

data = load_and_prepare_data()
analysis_vars = [c for c in data.columns if c != TARGET]
print("Analysis variables:", len(analysis_vars))



In [ ]:
# ============================================================
# Module 3. DirectLiNGAM estimation and DAG construction
# ============================================================

def fit_direct_lingam(df, threshold=LINGAM_EDGE_THRESHOLD, random_state=SEED):
    """
    Fit DirectLiNGAM and extract retained directed edges.
    In lingam adjacency_matrix_, B[i, j] means variable j -> variable i.
    """
    if not LINGAM_AVAILABLE:
        raise ImportError("The lingam package is required. Install it with: pip install lingam")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df.values)

    model = lingam.DirectLiNGAM(random_state=random_state)
    model.fit(X_scaled)

    B = pd.DataFrame(model.adjacency_matrix_, index=df.columns, columns=df.columns)

    rows = []
    for effect in B.index:
        for cause in B.columns:
            w = B.loc[effect, cause]
            if cause != effect and abs(w) >= threshold:
                rows.append({
                    "Cause": cause,
                    "Effect": effect,
                    "Weight": float(w),
                    "AbsWeight": float(abs(w))
                })

    edge_df = pd.DataFrame(rows).sort_values("AbsWeight", ascending=False).reset_index(drop=True)
    edge_df.to_csv(OUT_DIR / "Table_DirectLiNGAM_retained_edges.csv", index=False, encoding="utf-8-sig")

    return model, B, edge_df

def build_graph_from_edges(edge_df, nodes):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for _, r in edge_df.iterrows():
        G.add_edge(r["Cause"], r["Effect"], weight=r.get("Weight", 1.0))

    if not nx.is_directed_acyclic_graph(G):
        cycles = list(nx.simple_cycles(G))
        pd.DataFrame({"Cycle": [str(c) for c in cycles]}).to_csv(
            OUT_DIR / "Directed_cycles_detected.csv", index=False, encoding="utf-8-sig"
        )
        raise ValueError(
            "The retained graph is not acyclic. Increase LINGAM_EDGE_THRESHOLD or inspect Directed_cycles_detected.csv."
        )
    return G

lingam_model, B_lingam, edge_df = fit_direct_lingam(data, threshold=LINGAM_EDGE_THRESHOLD)
G = build_graph_from_edges(edge_df, nodes=data.columns)

dag_summary = pd.DataFrame({
    "Diagnostic": ["Is directed acyclic graph", "Number of nodes", "Number of retained edges"],
    "Result": [nx.is_directed_acyclic_graph(G), G.number_of_nodes(), G.number_of_edges()]
})
dag_summary.to_csv(OUT_DIR / "Table_DAG_acyclicity_summary.csv", index=False, encoding="utf-8-sig")
display(dag_summary)
display(edge_df.head(20))


In [ ]:
# ============================================================
# Module 4. DAG-derived adjustment-set construction
# ============================================================

from reviewer3_dag_helpers import derive_adjustment_table

adjustment_table = derive_adjustment_table(G, TREATMENTS, TARGET)
adjustment_table.to_csv(OUT_DIR / "Table_X_DAG_derived_adjustment_sets_FULL.csv", index=False, encoding="utf-8-sig")

table_x_clean = adjustment_table[[
    "Treatment",
    "Outcome",
    "Constructive generalized adjustment set",
    "Minimal DAG-derived adjustment set",
    "Expanded adjustment set for robustness",
    "Descendant-free generalized back-door set",
    "Adjustment-set derivation note"
]].copy()

table_x_clean.to_csv(OUT_DIR / "Table_X_DAG_derived_adjustment_sets_CLEAN.csv", index=False, encoding="utf-8-sig")

display(table_x_clean)



In [ ]:
# ============================================================
# Module 5. Visualisation of adjustment-set diagnostics
# Modified only for figure layout/style according to the requested format
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ast

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
})


def _find_col(df, keywords, exclude=None):
    """Find a column name containing all keywords, with optional excluded keywords."""
    exclude = exclude or []
    for col in df.columns:
        name = str(col).lower()
        if all(k.lower() in name for k in keywords) and not any(e.lower() in name for e in exclude):
            return col
    return None


def _count_set_cell(x):
    """
    Count variables stored as:
    1. numeric count, such as 14;
    2. list, tuple or set;
    3. string representation of list, tuple or set;
    4. comma/semicolon-separated string.

    This version keeps the original numeric counts unchanged.
    """
    if x is None:
        return 0

    if isinstance(x, (int, np.integer)):
        return int(x)

    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return 0
        return int(x)

    if isinstance(x, (list, tuple, set)):
        return len(x)

    s = str(x).strip()

    if s == "" or s.lower() in {"nan", "none", "[]", "{}", "set()"}:
        return 0

    # Try to parse string-form list/set/tuple first.
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple, set)):
            return len(obj)
        if isinstance(obj, (int, float, np.integer, np.floating)):
            return int(obj)
    except Exception:
        pass

    # Fallback: count comma/semicolon-separated items.
    for token in ["[", "]", "{", "}", "(", ")", "'", '"']:
        s = s.replace(token, "")

    parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]
    return len(parts)


def plot_adjustment_set_summary_revised(adjustment_table, palette, save_path):
    treatments = list(adjustment_table["Treatment"])

    candidate_col = _find_col(adjustment_table, ["candidate", "ancestor"])
    forbidden_col = _find_col(adjustment_table, ["forbidden"])
    desc_free_col = _find_col(adjustment_table, ["descendant", "free"])
    minimal_col = _find_col(adjustment_table, ["minimal"])
    constructive_col = _find_col(adjustment_table, ["constructive"])
    edge_removed_col = _find_col(adjustment_table, ["edge", "removed"])

    candidate_counts = [
        _count_set_cell(v) if candidate_col is not None else 0
        for v in (adjustment_table[candidate_col] if candidate_col is not None else [None] * len(adjustment_table))
    ]

    forbidden_counts = [
        _count_set_cell(v) if forbidden_col is not None else 0
        for v in (adjustment_table[forbidden_col] if forbidden_col is not None else [None] * len(adjustment_table))
    ]

    desc_free_counts = [
        _count_set_cell(v) if desc_free_col is not None else 0
        for v in (adjustment_table[desc_free_col] if desc_free_col is not None else [None] * len(adjustment_table))
    ]

    minimal_counts = [
        _count_set_cell(v) if minimal_col is not None else 0
        for v in (adjustment_table[minimal_col] if minimal_col is not None else [None] * len(adjustment_table))
    ]

    constructive_counts = [
        _count_set_cell(v) if constructive_col is not None else 0
        for v in (adjustment_table[constructive_col] if constructive_col is not None else [None] * len(adjustment_table))
    ]

    if edge_removed_col is not None:
        edge_removed = pd.to_numeric(
            adjustment_table[edge_removed_col],
            errors="coerce"
        ).fillna(0).to_numpy()
    else:
        edge_removed = np.zeros(len(adjustment_table), dtype=float)

    x = np.arange(len(treatments))

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15.8, 5.6),
        gridspec_kw={
            "width_ratios": [1.35, 1.35, 1.15],
            "wspace": 0.46
        }
    )

    fig.suptitle(
        "Formally derived adjustment sets from the estimated DirectLiNGAM graph",
        y=0.985,
        fontweight="bold",
        fontsize=16
    )

    # ============================================================
    # (a) Graph screening sets
    # ============================================================
    ax = axes[0]
    width = 0.34

    ax.bar(
        x - width / 2,
        candidate_counts,
        width=width,
        color=palette["blue"],
        edgecolor="black",
        linewidth=0.8,
        label="Candidate ancestors"
    )

    ax.bar(
        x + width / 2,
        forbidden_counts,
        width=width,
        color=palette["red"],
        edgecolor="black",
        linewidth=0.8,
        label="Forbidden nodes"
    )

    ax.set_ylabel("Number of nodes")
    ax.set_xticks(x)
    ax.set_xticklabels(treatments, rotation=22, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.28)
    ax.legend(frameon=True, loc="upper left")

    ax.text(
        0.5,
        -0.31,
        "(a) Graph screening sets",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=13,
        fontweight="normal"
    )

    # ============================================================
    # (b) Valid adjustment-set size
    # ============================================================
    ax = axes[1]
    width = 0.24

    ax.scatter(
        x - width,
        desc_free_counts,
        s=78,
        color=palette["purple"],
        edgecolor="black",
        linewidth=0.7,
        zorder=3,
        label="Descendant-free set"
    )

    ax.bar(
        x,
        minimal_counts,
        width=width,
        color=palette["teal"],
        edgecolor="black",
        linewidth=0.8,
        label="Minimal valid set"
    )

    ax.bar(
        x + width,
        constructive_counts,
        width=width,
        color=palette["orange"],
        edgecolor="black",
        linewidth=0.8,
        label="Canonical constructive set"
    )

    ax.set_ylabel("Number of covariates")
    ax.set_xticks(x)
    ax.set_xticklabels(treatments, rotation=22, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.28)
    ax.legend(frameon=True, loc="upper left")

    ax.text(
        0.5,
        -0.31,
        "(b) Valid adjustment-set size",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=13,
        fontweight="normal"
    )

    # ============================================================
    # (c) Proper back-door graph diagnostics
    # ============================================================
    ax = axes[2]
    y = np.arange(len(treatments))

    ax.hlines(
        y,
        0,
        edge_removed,
        color=palette["grey"],
        linewidth=1.8,
        alpha=0.85
    )

    ax.scatter(
        edge_removed,
        y,
        s=92,
        color=palette["green"],
        edgecolor="black",
        linewidth=0.8,
        zorder=3
    )

    # Only display Yes, not the full covariate names.
    for xi, yi in zip(edge_removed, y):
        ax.text(
            xi + 0.12,
            yi,
            "Descendant-free set: Yes",
            va="center",
            ha="left",
            fontsize=10,
            color=palette["green"]
        )

    ax.set_yticks(y)

    ax.set_yticklabels(
        treatments,
        rotation=45,
        ha="right",
        va="center",
        rotation_mode="anchor"
    )

    ax.tick_params(axis="y", pad=16)
    ax.set_xlabel("Edges removed from treatment in proper back-door graph")
    ax.grid(axis="x", linestyle="--", alpha=0.28)

    ax.set_xlim(
        left=-0.4,
        right=max(edge_removed) + 1.6 if len(edge_removed) else 1
    )

    # Keep the same vertical order as your original figure:
    # Daily sailing hours at the top and Ship course at the bottom.
    ax.invert_yaxis()

    ax.text(
        0.5,
        -0.31,
        "(c) Proper back-door graph diagnostics",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=13,
        fontweight="normal"
    )

    # ============================================================
    # Common style
    # ============================================================
    for ax in axes:
        ax.set_title("")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.07,
        right=0.99,
        top=0.84,
        bottom=0.28,
        wspace=0.46
    )

    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()


plot_adjustment_set_summary_revised(
    adjustment_table,
    palette=PALETTE,
    save_path=OUT_DIR / "Figure_D0_DAG_adjustment_set_summary.png"
)

In [ ]:
# ============================================================
# Module 6. DML-CF estimation under DAG-derived adjustment sets
# ============================================================

from reviewer3_dag_helpers import run_dml_cf_adjustment_robustness

if RUN_DML_CF:
    dml_robustness_table, effects_store = run_dml_cf_adjustment_robustness(
        data,
        adjustment_table,
        TREATMENTS,
        TARGET,
        seed=SEED,
        n_estimators=DML_MAIN_N_ESTIMATORS,
        min_samples_leaf=DML_MAIN_MIN_SAMPLES_LEAF,
    )
    dml_robustness_table.to_csv(
        OUT_DIR / "Table_DML_CF_adjustment_set_robustness.csv",
        index=False,
        encoding="utf-8-sig"
    )
    display(dml_robustness_table)
else:
    dml_robustness_table = pd.DataFrame()
    effects_store = {}
    print("DML-CF robustness analysis is disabled. Set RUN_DML_CF = True to run it.")



In [ ]:
# ============================================================
# Module 7. Visualisation of DML-CF robustness across adjustment sets
# ============================================================

from reviewer3_dag_helpers import plot_dml_adjustment_robustness

plot_dml_adjustment_robustness(
    dml_robustness_table,
    palette=PALETTE,
    save_path=OUT_DIR / "Figure_D1_DML_CF_adjustment_robustness.png"
)



In [ ]:
# ============================================================
# Module 8. Temporal block bootstrap for expanded DAG adjustment set
# Modified only for figure layout/style according to the requested format
# ============================================================

from reviewer3_dag_helpers import run_block_bootstrap_for_expanded_sets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy.stats import gaussian_kde
except Exception:
    gaussian_kde = None

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
})


def _get_boot_values(block_boot_store, treatment):
    """Robustly extract bootstrap ATE values from possible store formats."""
    item = block_boot_store.get(treatment, None) if isinstance(block_boot_store, dict) else None
    if item is None:
        return np.array([], dtype=float)
    if isinstance(item, pd.DataFrame):
        for col in ["ATE", "ate", "Effect", "effect"]:
            if col in item.columns:
                return pd.to_numeric(item[col], errors="coerce").dropna().to_numpy(dtype=float)
    if isinstance(item, dict):
        for key in ["ATE", "ate", "ates", "effects", "bootstrap_ates", "bootstrap_effects", "values"]:
            if key in item:
                return pd.to_numeric(pd.Series(item[key]), errors="coerce").dropna().to_numpy(dtype=float)
    return pd.to_numeric(pd.Series(item), errors="coerce").dropna().to_numpy(dtype=float)


def _plot_density_or_hist(ax, values, line_color, fill_color):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 3:
        ax.hist(values, bins=max(3, len(values)), density=True, color=fill_color,
                alpha=0.35, edgecolor="none")
        return
    if gaussian_kde is not None and np.nanstd(values) > 0:
        lo, hi = np.percentile(values, [1, 99])
        padding = (hi - lo) * 0.25 if hi > lo else 1.0
        xs = np.linspace(lo - padding, hi + padding, 250)
        ys = gaussian_kde(values)(xs)
        ax.fill_between(xs, ys, color=fill_color, alpha=0.35)
        ax.plot(xs, ys, color=line_color, linewidth=2.0)
    else:
        ax.hist(values, bins=20, density=True, color=fill_color, alpha=0.35, edgecolor="none")


def plot_block_bootstrap_distributions_revised(block_boot_store, palette, save_path):
    panel_order = ["Daily sailing hours", "Main engine RPM", "Daily sailing distance", "Ship course"]
    panel_labels = [
        "(a) Daily sailing hours",
        "(b) Main engine RPM",
        "(c) Daily sailing distance",
        "(d) Ship course",
    ]

    fig, axes = plt.subplots(2, 2, figsize=(12.8, 8.8))
    axes = axes.ravel()

    mean_handle = None
    interval_handle = None

    for ax, treatment, panel_label in zip(axes, panel_order, panel_labels):
        values = _get_boot_values(block_boot_store, treatment)
        _plot_density_or_hist(ax, values, line_color=palette["blue"], fill_color=palette["cyan"])

        if len(values) > 0:
            mean_value = float(np.nanmean(values))
            ci_low, ci_high = np.nanpercentile(values, [2.5, 97.5])
            interval_handle = ax.axvspan(ci_low, ci_high, color=palette["orange"], alpha=0.18,
                                         linestyle="--", linewidth=1.0, label="95% interval")
            ax.axvline(ci_low, color=palette["orange"], linestyle="--", linewidth=1.0)
            ax.axvline(ci_high, color=palette["orange"], linestyle="--", linewidth=1.0)
            mean_handle = ax.axvline(mean_value, color=palette["dark"], linewidth=1.7, label="Mean")

        ax.set_title("")
        ax.set_xlabel("ATE")
        ax.set_ylabel("Density")
        ax.grid(axis="both", linestyle="--", alpha=0.25)
        ax.text(0.5, -0.24, panel_label, transform=ax.transAxes,
                ha="center", va="top", fontsize=13, fontweight="normal")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle("Temporal block-bootstrap distribution under the canonical constructive adjustment set",
                 y=0.985, fontsize=16, fontweight="bold")

    # The legend is deliberately placed below the title and above the panels to avoid any overlap.
    handles = [h for h in [mean_handle, interval_handle] if h is not None]
    labels = ["Mean", "95% interval"][:len(handles)]
    if handles:
        fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.945),
                   ncol=len(handles), frameon=True)

    # Extra vertical and bottom space is reserved for the legend and non-bold subfigure labels.
    fig.subplots_adjust(left=0.075, right=0.985, top=0.875, bottom=0.115, hspace=0.43, wspace=0.25)
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()


if RUN_DML_CF:
    block_bootstrap_table, block_boot_store = run_block_bootstrap_for_expanded_sets(
        data,
        adjustment_table,
        TREATMENTS,
        TARGET,
        n_boot=BLOCK_BOOTSTRAP_N_REPS,
        block_size=BLOCK_BOOTSTRAP_BLOCK_SIZE,
        n_estimators=BLOCK_BOOTSTRAP_N_ESTIMATORS,
        min_samples_leaf=DML_MAIN_MIN_SAMPLES_LEAF,
        seed=SEED,
    )
    block_bootstrap_table.to_csv(
        OUT_DIR / "Table_temporal_block_bootstrap_DML_CF.csv",
        index=False,
        encoding="utf-8-sig"
    )
    display(block_bootstrap_table)
    plot_block_bootstrap_distributions_revised(
        block_boot_store,
        palette=PALETTE,
        save_path=OUT_DIR / "Figure_D2_temporal_block_bootstrap_DML_CF.png"
    )
else:
    block_bootstrap_table = pd.DataFrame()
    block_boot_store = {}



In [ ]:
# ============================================================
# Module 9. Automatic final reporting tables
# ============================================================

def build_final_reporting_summary(adjustment_table, dml_table=None, block_table=None):
    rows = []

    for _, r in adjustment_table.iterrows():
        treatment = r["Treatment"]
        base = {
            "Treatment": treatment,
            "Outcome": r["Outcome"],
            "Minimal DAG-derived adjustment set": r["Minimal DAG-derived adjustment set"],
            "Expanded robustness adjustment set": r["Expanded adjustment set for robustness"],
            "Adjustment-set note": r["Adjustment-set derivation note"]
        }

        if dml_table is not None and not dml_table.empty:
            sub = dml_table[
                (dml_table["Treatment"] == treatment) &
                (dml_table["Specification"] == "expanded_DAG_adjustment") &
                (dml_table["CV mode"] == "kfold") &
                (dml_table["Status"] == "Success")
            ]
            if not sub.empty:
                base["DML-CF ATE under expanded DAG adjustment"] = sub["ATE"].iloc[0]
            else:
                base["DML-CF ATE under expanded DAG adjustment"] = np.nan

        if block_table is not None and not block_table.empty:
            subb = block_table[block_table["Treatment"] == treatment]
            if not subb.empty and pd.notna(subb["ATE 2.5%"].iloc[0]):
                base["Temporal block-bootstrap 95% CI"] = f"[{subb['ATE 2.5%'].iloc[0]:.4f}, {subb['ATE 97.5%'].iloc[0]:.4f}]"
            else:
                base["Temporal block-bootstrap 95% CI"] = ""

        rows.append(base)

    summary = pd.DataFrame(rows)
    summary.to_csv(OUT_DIR / "Table_final_reporting_summary_for_manuscript.csv", index=False, encoding="utf-8-sig")
    return summary

final_summary_table = build_final_reporting_summary(
    adjustment_table,
    dml_robustness_table if "dml_robustness_table" in globals() else None,
    block_bootstrap_table if "block_bootstrap_table" in globals() else None
)

display(final_summary_table)

print("All outputs saved to:", OUT_DIR.resolve())
